# 05 train model
- Research candidates and evaluate approaches
- Decision matrix to shortlist candidates
- Implement baseline and challenger model with default/ basic settings
- Evaluate both models
- Hyperparameter tuning
- Production model selection

# Research:

## Candidate #1
### TF-IDF + Logistic Regression
##### TF-IDF:
Feature extraction technique used in NLP to transform raw text into a numeric value that ML models can process/ understand.
The numerical value is a weight to each word based on two factors:
1. How often a word appears in a document/ record (TF, Term frequency)
2. How rare the word is across all documents (IDF, Inverse document frequency)

This helps the model to focus on words that actually inform it on differences between records by stripping away common unhelpful words and highlighting distinctive words in each record so the model can learn differences.

Reference: https://scikit-learn.org/1.4/tutorial/text_analytics/working_with_text_data.html

##### Logistic regression:
Linear classification algorithm used for both binary and multi-class classification problems. Since our problem is multi-class (i.e. we have more than two possible outcomes/ labels) we can use scikit-learn multinominal logistic regression to estimate the probability of each possible class using a softmax function.

Using the TF-IDF transformed data, the model can then learn a weight for each feature (a word) and an associated class. We then select the class with the highest probability as the prediction.

During inference the model calculates the probability for each possible resolver group and the class with the highest probability is selected as the final predicted value.

Reference: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html

##### Overview of combination
TF-IDF + Logistic regression will be our baseline as it is computationally efficient, simple to explain and interpret and produces a probabalistic output to our multi-class classification problem.

Pros:
- Simple... basically text input, TF-IDF, Logistic regression = predicted resolver group. Low number of steps makes it easy to explain and implement
- Quick to train when compared with more complex neural network models so we can quickly experiment and iterate
- No black box processing, we can actually view the feature weights and see what words appear to be influencing classification decisions
- Probabilistic output allows us to route incidents for manual inspection if the confidence is low

Cons:
- TF-IDF just captures individual words, losing the meaning and the context of the words in the record
- Becuase each word is treated as a feature, the feature matrices can grow fairly large on a large number of records with lots of distinct words

Summary:
TF-IDF and logistic regression provide a good baseline which we can interpret and inspect as humans to understand it's behaviour. Where we may run into issues is the loss of context and meaning since TF-IDF treats individual words as features instead of phrases etc, we may find the model struggles with less obvious text descriptions or a large amount of variance in the words used for incidents of the same class.

To overcome these cons, other candidates will consider more advanced transformer based models.

____________________________________________________________________________________________________________________________________________

## Candidate #2
### DistilBERT - Transformer based language model
##### Transformer element
Transformers are a type of neural network design for processing natural language. The main difference between transformers and traditional NLP techniques like TF-IDF is that instead of treating individual words independently, transformers actually analyse the relationship between words in a sentence/ body of text using a mechanism called self-attention. This enables the model to understand not just words but the actual meaning of words in context.

References: 
- https://huggingface.co/docs/transformers/model_doc/distilbert
- https://www.ibm.com/think/topics/self-attention

##### DistilBERT
DistilBERT is a smaller and faster version of the BERT transformer model. It was created using a technique called knowledge distillation, which is where a smaller model is trained to replicate the behaviour of a larger pre-trained model. The result of that is a model that maintains a lot of the predictive power of the parent model but uses significantly less parameters and provides faster inference. This makes DistilBERT a good option when considering trade offs between performance/ computational cost.

References: 
- https://huggingface.co/docs/transformers/model_doc/bert
- https://huggingface.co/docs/transformers/model_doc/distilbert
- https://towardsdatascience.com/everything-you-need-to-know-about-albert-roberta-and-distilbert-11a74334b2da/

##### Overview of combination
DistilBERT can be tuned for a sequence classification task where the model directly predicts the resolver group based on the text input. Essentially text input, tokenizer, DistilBERT, classification head = predicted resolver group. The model outputs a probabilistic output similar to candidate #1, allowing us to select the highest probability as the class prediction.

Pros:
- Captures relationship between words to better understand context/ meaning
- Able to better understand variations in wording for the same incident type
- Often performs better on text classification tasks vs basic classifiers

Cons:
- Significantly more computationally expensive than traditional NLP models
- Requires additional dependencies like transformers and other libraries
- Harder to interpret compared with a linear model due to the complexity of the neural network

Summary:
DistilBERT will provide us with a more advanced approach to text classification by capturing relationships between words instead of just the individual words alone. This should in theory mean improved classification performance when the words used in descriptions vary. However, it will add complexity, longer training times and additional requirements for libraries/ infrastructure which can all be negatives.

____________________________________________________________________________________________________________________________________________

## Candidate #3
### SetFit - sentence transformer + classifier
##### Sentence transformer
Sentence transformers convert text into vector representations aka embeddings. These embeddings represent the meaning of a sentence rather than individual words, similar to candidate #2. This enables sentences with similar meanings to produce similar vector repsresentations so a model can make that link and use it to help classify a record.

References: 
- https://huggingface.co/docs/setfit/index
- https://sbert.net/index.html

##### SetFit
SetFit is a framework that combines sentence transformers with a classifier. Instead of full tuning a transformer model, setfit generates embeddings using a pre-trained transformer model and then trains a smaller classifier on top of those embeddings. This way it can reduce training time whilst still getting the benefit of the context and understanding provided by transformer models.

References: 
- https://huggingface.co/docs/setfit/index
- https://medium.com/@youssefchamrah/setfit-unpacked-when-sentence-transformers-go-to-gym-for-classification-muscle-56c16d9e69de

##### Overview of combination
SetFit looks simple when looking at it process-wise due to it's pre-trained nature. text input, sentence embeddings, classifier = predicted resolver group. It is also significantly faster than tuning a transformer model, again due to it's pre-trained nature.

Pros:
- Captures context and meaning using transformer embeddings
- Faster training compared with a full transformer tuning
- Lower computational cost than models like candidate #2

Cons:
- Introduces another modelling framework and dependency to the solution
- Less common to find implementations/ examples and research than the other candidates
- If there is lots of labelled training data, the benefit of the pre-trained nature is lowered significantly

Summary:
SetFit fits somewhere betweeen candidate #1 a traditional ML approach and candidate #2 a transformer based model. It can provide us with context and meaning similar to the transformer models whilst keeping training time lower due to it being pre-trained. However, if there is lots of training data which is labelled with the target we are trying to predict, it's benefit can be lessened significantly.

____________________________________________________________________________________________________________________________________________

# Decision Matrix
##### Candidates:
- Candidate 1: TF-IDF + Logsitic regression
- Candidate 2: DistilBERT
- Candidate 3: SetFit

##### Criteria:
- `Expected performance`: Likelihood of the model performing well on multi-class text classification
- `Ability to capture context and meaning` from text inputs: Our text descriptions are free text and will vary so meaning/ context will likely be very important
- `Explainability`: Need to be able to explain model behaviour to stakeholders
- `Complexity`: Trade off between complexity and performance with lower complexity preferred
- `Training and inference cost`: Keep cost and overhead low
- `Suitability for confidence based routing`: Must support probabalistic outputs so we can route tickets for manual review if confidence is low

##### Matrix:
| Criteria                                      | TF-IDF + Logistic regression | DistilBERT | SetFit      |
| ----------------------------------------------|----------------------------- | ---------- | ----------- |
| Expected performance                          | Medium                       | High       | Medium/High |
| Ability to capture context and meaning        | Low                          | High       | Medium/High |
| Explainability                                | High                         | Low        | Medium      |
| Complexity                                    | Low                          | High       | Medium      |
| Training and inference cost                   | Low                          | High       | Medium      |
| Suitability for confidence based routing      | High                         | High       | Medium/High |
| Fit for this project                          | High                         | High       | Medium      |

##### Decision:
1. TF-IDF + Logistic regression: Implement as a baseline model. This will provide us with a fast, simple and explainable baseline to reference and based on my research is a standard approach for classic text classification.
2. DistilBERT: Implement as the main challenger model vs our baseline model. It is a transformer model based on the powerful BERT model which based on research should yeild very strong performance.
3. SetFit: Not to be implemented for now. Although it may be an efficient way of approaching the problem when labelled data is limited, we have a significant amount of labelled data available for training so it makes less sense than tuning our own transformer based model on our specific data. 

Candidate 1 and 2 will be implemented for evaluation and comparison.

____________________________________________________________________________________________________________________________________________

# Load data and artefacts

In [9]:
# Imports
from pathlib import Path
from datetime import datetime
import io
import json
import yaml
import pandas as pd
from minio import Minio

In [10]:
# Load config from yaml
CONFIG_FILE = Path("../config/config.yaml")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Missing config file: {CONFIG_FILE.resolve()}")

with open(CONFIG_FILE, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

print("Config loaded")

Config loaded


In [11]:
# Read .env for credentials
def load_env_file(path: Path) -> dict:
    if not path.exists():
        raise FileNotFoundError(f"Missing env file: {path.resolve()}")

    env = {}
    for line in path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip()
    return env

# Env file location in repo
ENV_FILE = Path(f"{cfg['storage']['minio']['env_file']}")
# Storage endpoint and config
MINIO_ENDPOINT = cfg["storage"]["minio"]["endpoint"]
MINIO_SECURE = cfg["storage"]["minio"]["secure"]

# Authenticate to file storage list buckets to confirm connection
env = load_env_file(ENV_FILE)
client = Minio(
    MINIO_ENDPOINT,
    access_key=env["MINIO_ROOT_USER"],
    secret_key=env["MINIO_ROOT_PASSWORD"],
    secure=MINIO_SECURE,
)

print("Connected buckets:", [b.name for b in client.list_buckets()])

Connected buckets: ['data-profile-test', 'incident-pipeline', 'incident-pipeline-test', 'mlflow-artifacts']


In [18]:
# Load latest gold train, valid and test data from storage
gold_prefix_root = cfg["storage"]["minio"]["gold_training_prefix_root"].rstrip("/")
test_bucket = "incident-pipeline-test"

# Find latest timestamped gold run
run_tags = sorted({
    obj.object_name[len(gold_prefix_root) + 1:].split("/", 1)[0]
    for obj in client.list_objects(test_bucket, prefix=f"{gold_prefix_root}/", recursive=True)
    if obj.object_name.startswith(f"{gold_prefix_root}/") and len(obj.object_name.split("/")) >= 4
})

# Error if no gold runs found
if not run_tags:
    raise RuntimeError(f"No gold runs found under: {gold_prefix_root}")

# Use the latest gold run for training
latest_gold_run_tag = run_tags[-1]
latest_gold_run_prefix = f"{gold_prefix_root}/{latest_gold_run_tag}"

# Save this for model artifact traceability
training_data_run_tag = latest_gold_run_tag

# Paths for train, valid and test files
# Path for label mapping file
train_path = f"{latest_gold_run_prefix}/train.parquet"
valid_path = f"{latest_gold_run_prefix}/valid.parquet"
test_path = f"{latest_gold_run_prefix}/test.parquet"
label_mapping_path = f"{latest_gold_run_prefix}/label_mapping.json"

# Function to parquet files
def load_parquet_from_storage(bucket: str, object_name: str) -> pd.DataFrame:
    resp = client.get_object(bucket, object_name)
    try:
        return pd.read_parquet(io.BytesIO(resp.read()))
    finally:
        resp.close()
        resp.release_conn()

# Function to load json files
def load_json_from_storage(bucket: str, object_name: str) -> dict:
    resp = client.get_object(bucket, object_name)
    try:
        return json.loads(resp.read().decode("utf-8"))
    finally:
        resp.close()
        resp.release_conn()

# Files to dfs
train_df = load_parquet_from_storage(test_bucket, train_path)
valid_df = load_parquet_from_storage(test_bucket, valid_path)
test_df = load_parquet_from_storage(test_bucket, test_path)
label_mapping = load_json_from_storage(test_bucket, label_mapping_path)

# Print summary info on the run and loaded data
print("Latest gold run tag:", latest_gold_run_tag)
print("Latest gold prefix:", latest_gold_run_prefix)
print("Train rows:", len(train_df))
print("Valid rows:", len(valid_df))
print("Test rows:", len(test_df))
print("Label mapping size:", len(label_mapping["classes"]))


Latest gold run tag: 20260305_212830
Latest gold prefix: gold/training/20260305_212830
Train rows: 7999
Valid rows: 1001
Test rows: 1000
Label mapping size: 12


In [19]:
# Label mapping check
display(label_mapping["classes"])

['App Support - ERP',
 'App Support - Finance',
 'App Support - HRIS',
 'App Support - M365',
 'App Support - Microsoft Fabric',
 'App Support - Power BI',
 'App Support - Power Platform',
 'End User Compute',
 'Identity and User Access',
 'Integration & Middleware',
 'Network Ops',
 'Security Operations']

# Validate data

In [ ]:
# validate columns are present in all splits and check for leakage
# required cols
required_cols = ["sys_id", "text", "label_final", "label_id"]

# check for required cols in each split
for name, df in [("train", train_df), ("valid", valid_df), ("test", test_df)]:
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{name} is missing required columns: {missing}")

    # Print now number, null rate, empty rate and number of labels in each split
    print(f"\n{name.upper()} checks")
    print("Rows:", len(df))
    print("Empty text rate:", f"{df['text'].fillna('').astype(str).str.strip().eq('').mean():.2%}")
    print("Null label rate:", f"{df['label_final'].isna().mean():.2%}")
    print("Unique labels:", df["label_final"].nunique())

# Collect sys_ids for leakage check
train_ids = set(train_df["sys_id"])
valid_ids = set(valid_df["sys_id"])
test_ids = set(test_df["sys_id"])

# Print overlap of sys_ids between splits to check for leakage
print("\nSplit leakage")
print("Train/Valid overlap:", len(train_ids & valid_ids))
print("Train/Test overlap :", len(train_ids & test_ids))
print("Valid/Test overlap :", len(valid_ids & test_ids))


TRAIN checks
Rows: 7999
Empty text rate: 0.00%
Null label rate: 0.00%
Unique labels: 12

VALID checks
Rows: 1001
Empty text rate: 0.00%
Null label rate: 0.00%
Unique labels: 12

TEST checks
Rows: 1000
Empty text rate: 0.00%
Null label rate: 0.00%
Unique labels: 12

Split leakage checks
Train/Valid overlap: 0
Train/Test overlap : 0
Valid/Test overlap : 0


# Implementation candidate #1
TF-IDF + Logistic regression